In [1]:
#Start

# !/usr/bin/env python
# coding: utf-8

# Importing Pandas Library as variable pd
import pandas as pd
from pathlib import Path

DATA_DIR = Path("../data")
OUTPUT_DIR = Path("../outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

In [2]:
# Import the data from the CSV file into a data frame

df = pd.read_csv(DATA_DIR / "employee_turnover.csv")

In [3]:
# Print column names for review
print(df.columns)

Index(['EmployeeNumber', 'Age', 'Tenure', 'Turnover', 'HourlyRate ',
       'HoursWeekly', 'CompensationType', 'AnnualSalary',
       'DrivingCommuterDistance', 'JobRoleArea', 'Gender', 'MaritalStatus',
       'NumCompaniesPreviouslyWorked', 'AnnualProfessionalDevHrs',
       'PaycheckMethod', 'TextMessageOptIn'],
      dtype='object')


Reviewing the columns, I have noticed an error in the column name, HourlyRate, it has a space located at the end. I don't know if this will have major implications on our dataset but it is likely best to fix that early for easy querying and analysis later on. 'HourlyRate '

In [4]:
# Review characteristics of the data

# shape[0] returns total rows
# shape[1] returns total columns
print(f"There are {df.shape[0]} rows and {df.shape[1]} columns in the dataset")

There are 10199 rows and 16 columns in the dataset


In [5]:
# Review data types of each column
df_types = df.dtypes
print(df_types)

EmployeeNumber                    int64
Age                               int64
Tenure                            int64
Turnover                         object
HourlyRate                       object
HoursWeekly                       int64
CompensationType                 object
AnnualSalary                    float64
DrivingCommuterDistance           int64
JobRoleArea                      object
Gender                           object
MaritalStatus                    object
NumCompaniesPreviouslyWorked    float64
AnnualProfessionalDevHrs        float64
PaycheckMethod                   object
TextMessageOptIn                 object
dtype: object


After reviewing the data types for this dataframe, I have noticed another issue with the HourlyRate column. The data type appears to return as an object, and/or text/string. This should be corrected to a numeric for analysis.

In [6]:
#Review examples from each column
print(df.head(4))

   EmployeeNumber  Age  Tenure Turnover HourlyRate   HoursWeekly  \
0               1   28       6      Yes     $24.37            40   
1               2   33       2      Yes     $24.37            40   
2               3   22       1       No     $22.52            40   
3               4   23       1       No     $22.52            40   

  CompensationType  AnnualSalary  DrivingCommuterDistance  \
0           Salary       50689.6                       89   
1           Salary       50689.6                       89   
2           Salary       46841.6                       35   
3           Salary       46841.6                       35   

              JobRoleArea  Gender MaritalStatus  NumCompaniesPreviouslyWorked  \
0                Research  Female       Married                           3.0   
1                Research  Female       Married                           6.0   
2  Information_Technology  Female        Single                           1.0   
3  Information_Technology  Fe

Upon the review of samples, I noticed the column HourlyRate also appears to return the symbol "$" within the entry. This could also be an issue for later analysis. I have also seen a discrepancy with the results from the PaycheckMethod column as the results incidate some users input "Mail Check" vs "Mailed Check" as well as missing data from AnnualProfessionalDevHrs.

### Duplicate Entries Inspection

In [7]:
# Duplicate Entries

print(f'There is currently {df.duplicated().sum()} duplicate rows in the dataset.')


There is currently 99 duplicate rows in the dataset.


In [8]:
# Fixing Duplicate Entries

df = df.drop_duplicates()  # drops all duplicate rows

print(f'There is now {df.duplicated().sum()} duplicate rows in the dataset after removing duplicates.')



There is now 0 duplicate rows in the dataset after removing duplicates.


### Missing Values Inspection

In [9]:
#  Checking for missing values

print(df.isnull().sum())

print(f'\nThere is currently {df.isnull().sum().sum()} missing values in the dataset.')

print(f'The dataset contains {df.shape[0]} rows.')



EmployeeNumber                     0
Age                                0
Tenure                             0
Turnover                           0
HourlyRate                         0
HoursWeekly                        0
CompensationType                   0
AnnualSalary                       0
DrivingCommuterDistance            0
JobRoleArea                        0
Gender                             0
MaritalStatus                      0
NumCompaniesPreviouslyWorked     663
AnnualProfessionalDevHrs        1947
PaycheckMethod                     0
TextMessageOptIn                2258
dtype: int64

There is currently 4868 missing values in the dataset.
The dataset contains 10100 rows.


In [10]:
# Dropping Missing Values
df = df.dropna()

print(f'\nThere is now {df.isnull().sum().sum()} missing values in the dataset after removing missing values.')

print(f'The dataset now contains {df.shape[0]} rows.')


There is now 0 missing values in the dataset after removing missing values.
The dataset now contains 5917 rows.


Rationale notes:

Number of Companies Worked - The employee's company work history
    
    Dropped Reason: Management needs to know how many companies the employee worked at, this matters as this data could skew strategy regarding whether the issue is the company, or the employee's habits.

Annual Professional Development Hours - The number of hours employees spend on professional development.
    
    Dropped Reason: Management needs to know whether the employee was possibly upskilling to progress in their career as this could have an impact on turnover.

Text Message Opt-In - Employees who receive text messages related to employer information.
    
    Dropped Reason: I did not believe this was as significant to the impact of turnover but communication lines are crucial in business, and it is possible that an employee was alienated and this reduced their workplace satisfaction.

### Inconsistent Entries Inspection

In [11]:
df.nunique()

EmployeeNumber                  5917
Age                               41
Tenure                            20
Turnover                           2
HourlyRate                      3871
HoursWeekly                        1
CompensationType                   1
AnnualSalary                    4029
DrivingCommuterDistance          120
JobRoleArea                       12
Gender                             3
MaritalStatus                      3
NumCompaniesPreviouslyWorked       9
AnnualProfessionalDevHrs          21
PaycheckMethod                     7
TextMessageOptIn                   2
dtype: int64

In comparison to our data dictionary, the following has been observed from each column according to the uniqueness:

EmployeeNumber - This number must be unique to identify the employee. This appears fine for this inspection.

Age - This should be unique as all employees will not be the same age but is likely within a range.

Tenure - Employees will not all have worked the same amount of years, but will also be within a range.

Turnover -  There are two values, "yes, no" and there were two unique values returned. This appears fine.

HourlyRate - It is not uncommon for there to be many different hourly rates among the employees depending on various factors.

HoursWeekly - Given our sampling, and unique return value, it appears that all employees work 40 hours per week.

CompensationType - With the observable sampling and unique value, it appears that all employees are paid on a salary basis.

AnnualSalary - As previoulsy observed with HourlyRate, we see that employee compensation rates do vary.

DrivingCommuterDistance - There are 120 unique values, while most employees do likely live close so it is also likely within a range.

JobRoleArea - There currently seems to be 12 different areas of job role areas. The dictionary given doesn't indicate all possibilities so this will need to be further inspected.

Gender - According to the dictionary, there is only three options, "male, female, perfer not to answer". This appears fine.

MartialStatus - According to the dictionary, there is three options, "single, divorced, married". This appears fine.

NumCompaniesPreviouslyWorked - A different/unqiue amount of companies worked at doesn't currently seem unordinary.

AnnualProfessionalDevHrs - A different/unique number of hours spent on professional development doesn't seem out of the ordinary.

PaycheckMethod - There currently seem to be 7 different ways to be paid. The dictionary given doesn't indicate all possibilities so this will need to be further inspected.

TextMessageOptIn - From the dictionary and sampling, it is concluded that there are only two options. This appears fine.

### Formating Errors

In [12]:
#Review more examples from each column
print(df.sample(4))

      EmployeeNumber  Age  Tenure Turnover HourlyRate   HoursWeekly  \
8363            8364   31       2       No     $30.16            40   
4574            4575   53      14       No     $76.04            40   
6484            6485   36       7       No     $93.51            40   
2310            2311   21       1      Yes     $20.40            40   

     CompensationType  AnnualSalary  DrivingCommuterDistance  \
8363           Salary       62732.8                      -13   
4574           Salary      158163.2                       72   
6484           Salary      334500.8                       95   
2310           Salary       42432.0                      250   

                 JobRoleArea  Gender MaritalStatus  \
8363  Information Technology  Female        Single   
4574                Research  Female       Married   
6484         Human Resources    Male      Divorced   
2310           Manufacturing    Male        Single   

      NumCompaniesPreviouslyWorked  AnnualProfession

After reviewing the potential uniqueness of each column’s data, additional samples are printed to further dial in the accuracy of the data scope.

In [13]:
# Check the amount of job role areas in the dataset

unique_jobs = df['JobRoleArea'].unique()
print(f"Unique values in {'JobRoleArea'}: {unique_jobs}\n")

Unique values in JobRoleArea: ['Research' 'Information_Technology' 'Human_Resources' 'Laboratory'
 'Sales' 'Manufacturing' 'Healthcare' 'Marketing' 'InformationTechnology'
 'HumanResources' 'Information Technology' 'Human Resources']



There appears to be the following job role areas:

1. Research
2. Information Technology
3. Sales
4. Human Resources
5. Laboratory
6. Manufacturing
7. Healthcare
8. Marketing

These entries will need to be formatted for consistency.

In [14]:
# Standardizing JobRoleArea values
df['JobRoleArea'] = df['JobRoleArea'].replace({
    'Information_Technology': 'Information Technology',
    'InformationTechnology': 'Information Technology',
    'Human_Resources': 'Human Resources',
    'HumanResources': 'Human Resources'
})

# Verify the unique values again
unique_jobs = df['JobRoleArea'].unique()
print(f"Unique values in 'JobRoleArea': {unique_jobs}\n")


Unique values in 'JobRoleArea': ['Research' 'Information Technology' 'Human Resources' 'Laboratory'
 'Sales' 'Manufacturing' 'Healthcare' 'Marketing']



In [15]:
# Check the amount of PaycheckMethods in the dataset

unique_pay = df['PaycheckMethod'].unique()
print(f"Unique values in {'PaycheckMethod'}: {unique_pay}\n")

Unique values in PaycheckMethod: ['Mail Check' 'Mailed Check' 'Direct_Deposit' 'DirectDeposit'
 'Direct Deposit' 'Mail_Check' 'MailedCheck']



There appears to be the following paycheck methods available:

1. Mail Check
2. Direct Deposit

These will also need to be formatted for consistency.

In [16]:
# Standardizing PaycheckMethod values
df['PaycheckMethod'] = df['PaycheckMethod'].replace({
    'Mail_Check': 'Mail Check',
    'MailedCheck': 'Mail Check',
    'Mailed Check': 'Mail Check',
    'Direct_Deposit': 'Direct Deposit',
    'DirectDeposit': 'Direct Deposit'
})

# Verify the unique values again
unique_pay = df['PaycheckMethod'].unique()
print(f"Unique values in 'PaycheckMethod': {unique_pay}\n")


Unique values in 'PaycheckMethod': ['Mail Check' 'Direct Deposit']



In [17]:
# Review column name to ensure previously mentioned format error
print(df.columns)

Index(['EmployeeNumber', 'Age', 'Tenure', 'Turnover', 'HourlyRate ',
       'HoursWeekly', 'CompensationType', 'AnnualSalary',
       'DrivingCommuterDistance', 'JobRoleArea', 'Gender', 'MaritalStatus',
       'NumCompaniesPreviouslyWorked', 'AnnualProfessionalDevHrs',
       'PaycheckMethod', 'TextMessageOptIn'],
      dtype='object')


There should be a fix to the column name "HourlyRate " as this indicates there is a space afterwards.

In [18]:
# Inpsect the column entries further to identify any format issues
print(f'The column, HourlyRate is currently formatted as a:',df['HourlyRate '].dtype, '\n')

#Show more samples of the HourlyRate column
print(df['HourlyRate '].sample(10))

The column, HourlyRate is currently formatted as a: object 

4487    $80.23 
1036    $54.80 
2966    $30.00 
8683    $57.65 
1787    $28.28 
2131    $28.47 
3718    $34.74 
2030    $78.51 
9372    $81.32 
297     $91.80 
Name: HourlyRate , dtype: object


Since the column HourlyRate returns as an object, it is assumed to be in a string format here, and given the samples, it appears this could be due to the denomination symbol "$" within each entry.

In [19]:
# Prove if the format issue with the current column data

# Attempt to sum just a few entries to check the format issue
print(df['HourlyRate '].head(5).sum())  # If it concatenates instead of summing, it's a string

$24.37 $24.37 $22.52 $28.43 $28.43 


The data will not sum, so it must be reformatted to a numeric, float data type. The denomination symbol will also be removed or added to the column name to ensure there will be no further issues down the line.

In [20]:
# Fix trailing space in column name
df.rename(columns={'HourlyRate ': 'HourlyRate'}, inplace=True)

# Remove the "$" symbol and convert the 'HourlyRate ' column to numeric
df['HourlyRate'] = df['HourlyRate'].replace(r'[\$,]', '', regex=True).astype(float)

# Check the column data type to ensure the conversion was successful
print(df['HourlyRate'].sample(10))


1187    35.28
4957    28.59
766     67.40
1912    17.64
6907    51.36
1255    84.60
9259    64.33
5027    50.90
7447    41.79
8814    31.08
Name: HourlyRate, dtype: float64


In [21]:
# Check the sum of the first 5 entries to ensure the format issue was fixed

print(df['HourlyRate'].head(5).sum())  # If it concatenates instead of summing, it's a string

128.12


#### Outliers

In [22]:
# Inspect the columns for outliers
df.describe()

,EmployeeNumber,Age,Tenure,HourlyRate,HoursWeekly,AnnualSalary,DrivingCommuterDistance,NumCompaniesPreviouslyWorked,AnnualProfessionalDevHrs
count,5917.000000,5917.000000,5917.000000,5917.000000,5917.0,5917.000000,5917.000000,5917.000000,5917.000000
mean,4797.132669,43.874768,8.904512,51.742804,40.0,117706.387054,46.133176,4.242353,14.873754
std,2981.834839,10.100210,5.491204,23.679259,0.0,75819.370757,54.498238,2.487918,6.080132
min,1.000000,21.000000,1.000000,17.210000,40.0,-33326.400000,-275.000000,1.000000,5.000000
25%,2049.000000,37.000000,5.000000,30.810000,40.0,63273.600000,13.000000,2.000000,10.000000
50%,4717.000000,43.000000,8.000000,45.680000,40.0,95014.400000,42.000000,4.000000,15.000000
75%,7414.000000,52.000000,13.000000,72.110000,40.0,149988.800000,71.000000,6.000000,20.000000
max,10099.000000,61.000000,20.000000,98.050000,40.0,339950.400000,950.000000,9.000000,25.000000


In [23]:
# Drop the negative values in the 'AnnualSalary' and 'DrivingCommuterDistance' columns
df = df.loc[(df['AnnualSalary'] >= 0.0) & (df['DrivingCommuterDistance'] >= 0.0)]

In [24]:
# Check the columns for outliers again
df.describe()

,EmployeeNumber,Age,Tenure,HourlyRate,HoursWeekly,AnnualSalary,DrivingCommuterDistance,NumCompaniesPreviouslyWorked,AnnualProfessionalDevHrs
count,5088.000000,5088.000000,5088.000000,5088.000000,5088.0,5088.000000,5088.000000,5088.000000,5088.000000
mean,4815.447917,43.950472,8.901533,51.945206,40.0,118688.720401,55.364780,4.248624,14.933962
std,2981.148586,10.085749,5.471726,23.638798,0.0,75213.334131,51.571546,2.493810,6.084521
min,1.000000,21.000000,1.000000,17.210000,40.0,1432.000000,0.000000,1.000000,5.000000
25%,2093.500000,37.000000,5.000000,30.927500,40.0,63835.200000,27.000000,2.000000,10.000000
50%,4738.500000,43.000000,8.000000,46.260000,40.0,96148.000000,50.000000,4.000000,15.000000
75%,7436.500000,52.000000,13.000000,72.262500,40.0,150325.400000,75.000000,6.000000,20.000000
max,10096.000000,61.000000,20.000000,98.050000,40.0,339950.400000,950.000000,9.000000,25.000000


In [25]:
# Select only numeric columns
numeric_columns = df.select_dtypes(include=['number']).columns

# Calculate IQR for each numeric column
Q1 = df[numeric_columns].quantile(0.25)
Q3 = df[numeric_columns].quantile(0.75)
IQR = Q3 - Q1

# Define outlier bounds
lower_bound = Q1 - 3.0 * IQR
upper_bound = Q3 + 3.0 * IQR

# Find outliers
outliers = (df[numeric_columns] < lower_bound) | (df[numeric_columns] > upper_bound)

# Count the number of outliers per column
outlier_counts = outliers.sum()

print(outlier_counts)



EmployeeNumber                    0
Age                               0
Tenure                            0
HourlyRate                        0
HoursWeekly                       0
AnnualSalary                      0
DrivingCommuterDistance         143
NumCompaniesPreviouslyWorked      0
AnnualProfessionalDevHrs          0
dtype: int64


In [26]:
# Filter dataset: Keep only rows where selected columns are within bounds
df = df[~((df[numeric_columns] < lower_bound) | (df[numeric_columns] > upper_bound)).any(axis=1)]

# Verify if outliers are removed
outlier_counts = ((df[numeric_columns] < lower_bound) | (df[numeric_columns] > upper_bound)).sum()
print(outlier_counts)



EmployeeNumber                  0
Age                             0
Tenure                          0
HourlyRate                      0
HoursWeekly                     0
AnnualSalary                    0
DrivingCommuterDistance         0
NumCompaniesPreviouslyWorked    0
AnnualProfessionalDevHrs        0
dtype: int64


In [27]:
# Inspect the columns for outliers once again
df.describe()

,EmployeeNumber,Age,Tenure,HourlyRate,HoursWeekly,AnnualSalary,DrivingCommuterDistance,NumCompaniesPreviouslyWorked,AnnualProfessionalDevHrs
count,4945.000000,4945.000000,4945.000000,4945.000000,4945.0,4945.000000,4945.000000,4945.000000,4945.000000
mean,4905.835592,43.972093,8.917290,51.939001,40.0,118766.130354,48.857027,4.255005,14.948837
std,2970.889662,10.105993,5.477801,23.649220,0.0,75369.343687,28.105319,2.494551,6.092963
min,1.000000,21.000000,1.000000,17.210000,40.0,1432.000000,0.000000,1.000000,5.000000
25%,2217.000000,37.000000,5.000000,30.910000,40.0,63835.200000,27.000000,2.000000,10.000000
50%,4907.000000,44.000000,8.000000,46.250000,40.0,96096.000000,49.000000,4.000000,15.000000
75%,7515.000000,52.000000,13.000000,72.350000,40.0,150488.000000,73.000000,6.000000,20.000000
max,10096.000000,61.000000,20.000000,98.050000,40.0,339950.400000,125.000000,9.000000,25.000000


In [28]:
#Ensure distribution of data is normal

for column in df.columns:
    print(f"Value counts for {column}:")
    print(df[column].value_counts())
    print("\n")

Value counts for EmployeeNumber:
EmployeeNumber
1        1
6630     1
6646     1
6645     1
6644     1
        ..
3113     1
3112     1
3110     1
3108     1
10096    1
Name: count, Length: 4945, dtype: int64


Value counts for Age:
Age
39    241
37    238
36    214
38    203
40    182
43    157
44    153
46    151
60    149
61    147
48    146
56    146
47    143
42    142
57    142
59    140
53    135
52    135
54    134
58    132
45    132
41    131
51    128
49    127
50    123
55    111
34    109
33    104
32    102
30    102
35     93
31     90
25     49
22     46
24     44
29     43
28     41
23     38
26     35
27     35
21     32
Name: count, dtype: int64


Value counts for Tenure:
Tenure
1     412
7     358
6     355
5     355
9     355
8     353
10    349
3     301
2     236
4     230
14    180
16    177
15    170
17    165
13    164
19    163
18    161
11    161
20    158
12    142
Name: count, dtype: int64


Value counts for Turnover:
Turnover
No     2643
Yes    2302
Name:

HourlyRate
31.20    7
34.28    7
33.29    6
28.53    6
33.77    6
        ..
40.06    1
61.92    1
81.37    1
45.04    1
93.05    1
Name: count, Length: 3436, dtype: int64


Value counts for HoursWeekly:
HoursWeekly
40    4945
Name: count, dtype: int64


Value counts for CompensationType:
CompensationType
Salary    4945
Name: count, dtype: int64


Value counts for AnnualSalary:
AnnualSalary
64896.0     7
337516.8    6
70241.6     6
334729.6    5
47507.2     5
           ..
117166.4    1
134867.2    1
92726.4     1
162865.6    1
333544.0    1
Name: count, Length: 3545, dtype: int64


Value counts for DrivingCommuterDistance:
DrivingCommuterDistance
33     97
28     96
42     77
77     68
66     63
       ..
95     32
90     29
23     28
91     26
125    13
Name: count, Length: 99, dtype: int64


Value counts for JobRoleArea:
JobRoleArea
Research                  978
Sales                     947
Manufacturing             522
Healthcare                515
Information Technology    512
Ma

In [29]:
# Check the correlation of hourlyrate and annualsalary
correlation = df['HourlyRate'].corr(df['AnnualSalary'])
print(f"Correlation between HourlyRate and AnnualSalary is : {correlation}")

initial_row_count = df.shape[0]
print(f"Number of rows before cleaning due to correlation: {initial_row_count}")

Correlation between HourlyRate and AnnualSalary is : 0.9142574233552186
Number of rows before cleaning due to correlation: 4945


In [30]:

# Drops rows where the AnnualSalary is not within 50% of the calculated AnnualSalary
df = df[abs(df['AnnualSalary'] - (df['HourlyRate'] * 2080)) / (df['HourlyRate'] * 2080) <= 0.5]


print(f"Correlation after cleanup: {df['HourlyRate'].corr(df['AnnualSalary'])}")
final_row_count = df.shape[0]  # Number of rows after cleaning
rows_removed = initial_row_count - final_row_count
print(f"Number of rows removed due to correlation: {rows_removed}")
print(f"Number of rows after cleaning due to correlation: {final_row_count}")

Correlation after cleanup: 0.9969548522514307
Number of rows removed due to correlation: 481
Number of rows after cleaning due to correlation: 4464


In [31]:
# Check the distribution and counts of data after cleaning

for column in df.columns:
    print(f"Value counts for {column}:")
    print(df[column].value_counts())
    print("\n")

Value counts for EmployeeNumber:
EmployeeNumber
1        1
6623     1
6645     1
6644     1
6639     1
        ..
3170     1
3176     1
3177     1
3178     1
10096    1
Name: count, Length: 4464, dtype: int64


Value counts for Age:
Age
39    225
37    220
36    197
38    190
40    169
44    136
48    134
46    133
42    128
43    128
61    127
60    127
56    126
59    126
57    123
52    120
47    119
45    118
53    117
49    115
41    115
58    115
51    114
54    111
34    109
33    104
50    103
30    102
32    102
55     95
35     93
31     90
25     44
29     43
24     41
22     41
28     41
27     34
23     33
26     29
21     27
Name: count, dtype: int64


Value counts for Tenure:
Tenure
1     384
5     326
6     325
9     323
7     320
8     317
10    315
3     275
4     220
2     219
14    155
16    153
15    152
18    144
19    142
11    142
17    141
13    141
20    141
12    129
Name: count, dtype: int64


Value counts for Turnover:
Turnover
No     2386
Yes    2078
Name:

In [32]:
df.describe()

,EmployeeNumber,Age,Tenure,HourlyRate,HoursWeekly,AnnualSalary,DrivingCommuterDistance,NumCompaniesPreviouslyWorked,AnnualProfessionalDevHrs
count,4464.000000,4464.000000,4464.000000,4464.000000,4464.0,4464.000000,4464.000000,4464.000000,4464.000000
mean,4911.761425,43.616039,8.801075,48.126555,40.0,99838.563441,48.752016,4.222446,14.972894
std,2963.998215,10.104317,5.466402,20.928409,0.0,44050.827823,28.046017,2.491757,6.100248
min,1.000000,21.000000,1.000000,17.340000,40.0,36067.200000,0.000000,1.000000,5.000000
25%,2230.250000,36.000000,5.000000,30.220000,40.0,62398.000000,27.000000,2.000000,10.000000
50%,4918.500000,43.000000,8.000000,42.730000,40.0,88878.400000,48.000000,4.000000,15.000000
75%,7519.500000,52.000000,13.000000,65.520000,40.0,136281.600000,72.000000,6.000000,20.000000
max,10096.000000,61.000000,20.000000,98.050000,40.0,203944.000000,125.000000,9.000000,25.000000


In [33]:
# Cleaned Dataset CSV Output
df.to_csv(OUTPUT_DIR / "employee_turnover_cleaned.csv", index=False)